# 3D CNN (ResNet-18) Trainer
- Use this notebook file to directly train from the video dataset without conversion to images
- Trains based on ResNet-18 3D architecture CNN (This allows direct video data training)

### Library Imports

This cell imports all the necessary libraries for building and training a video classification model.

- `torch`, `torch.nn`, `torch.optim`: PyTorch libraries for building neural networks, defining layers, and optimizing models.
- `torch.utils.data`: Provides utilities for working with datasets and data loaders.
- `torchvision.transforms`: Common image transformations (will be adapted for video).
- `torchvision.models.video.r3d_18`: Imports the R3D-18 pre-trained video model.
- `decord`, `decord.cpu`: Library for efficient video loading and decoding, utilizing the CPU.
- `numpy`: For numerical operations.
- `os`: For interacting with the operating system (e.g., path manipulation).
- `sklearn.model_selection.train_test_split`: For splitting data into training, validation, and testing sets.
- `torch.nn.functional`: Provides functional interfaces for neural network operations (like interpolation).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.video import r3d_18
from decord import VideoReader, cpu
import numpy as np
import os
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
from sklearn.utils import resample
import matplotlib.pyplot as plt
from datetime import datetime

### Specify Dataset Root

This cell defines the absolute path to the root directory of the violence detection dataset. **Make sure to adjust this path to match the location of the dataset on your system.**

In [ ]:
# Specify dataset root with absolute path (adjust to your exact location)
DATA_ROOT = '/home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset'  # Example; change if different

### Define Class Directories

This cell constructs the absolute paths to the 'violent' and 'non-violent' subdirectories within the `DATA_ROOT`. These directories are expected to contain further subdirectories (like `cam1` and `cam2`) holding the video files.

In [ ]:
# Class-specific directories (each contains cam1/ and cam2/ with MP4s)
VIOLENT_DIR = os.path.join(DATA_ROOT, 'violent')
NON_VIOLENT_DIR = os.path.join(DATA_ROOT, 'non-violent')

### Verify Directory Existence

This cell performs a quick check to confirm that the specified `DATA_ROOT`, `NON_VIOLENT_DIR`, and `VIOLENT_DIR` paths actually exist on the filesystem. It prints a confirmation or a warning if a directory is not found. This helps in debugging path issues early on.

In [ ]:
# Quick check for directory existence
print(f"Current notebook working directory: {os.getcwd()}")
for dir_path in [DATA_ROOT, NON_VIOLENT_DIR, VIOLENT_DIR]:
    if os.path.exists(dir_path):
        print(f"{dir_path} exists.")
    else:
        print(f"WARNING: {dir_path} does NOT exist. Check your path!")

Current notebook working directory: /home/heytanix/Documents/Jain_PCL/PCL_repository
/home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset exists.
/home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/non-violent exists.
/home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/violent exists.


### Define CSV File Paths

This cell defines the paths to the CSV files included with the dataset. While these files might contain additional information about action classes and occurrences, they are commented out in this specific notebook and are not used for the binary classification task. They are included here for reference.

In [ ]:
# CSV files (for reference; not used in binary classification but can be loaded if needed)
VIOLENT_CLASSES_CSV = os.path.join(DATA_ROOT, 'violent-action-classes.csv')
NONVIOLENT_CLASSES_CSV = os.path.join(DATA_ROOT, 'nonviolent-action-classes.csv')
ACTION_OCCURRENCES_CSV = os.path.join(DATA_ROOT, 'action-class-occurrences.csv')

### Define Class Labels

This cell defines a list of strings representing the class labels for the classification task. These labels correspond to the names of the dataset directories.

In [ ]:
# Class labels
CLASSES = ['non-violent', 'violent']  # Match your folder names

### Find MP4 Files and Split Data

This cell defines a function `find_mp4_files` to recursively search for and collect all `.mp4` files (case-insensitive) within a given directory. It includes debugging prints to show which directories are being searched and which files are found.

After defining the function, it calls it for both the violent and non-violent directories to get lists of video paths. It then assigns numerical labels (0 for non-violent, 1 for violent) to each video path and combines them into a single list.

A check is performed to ensure that MP4 files were found; otherwise, a `ValueError` is raised with helpful debugging tips.

Finally, the cell uses `sklearn.model_selection.train_test_split` to split the data into training, validation, and testing sets. Stratification is used to ensure that the proportion of violent and non-violent videos is maintained in each split. The data is split into 85% for training+validation and 15% for testing, and then the training+validation set is further split into training (approx 70%) and validation (approx 15%). The number of samples in each split is printed for verification.

In [ ]:
# Function to recursively find MP4 files in a directory (case-insensitive extension, with debugging prints)
def find_mp4_files(directory):
    mp4_files = []
    if not os.path.exists(directory):
        print(f"Directory {directory} does not exist. Skipping.")
        return mp4_files
    for root, _, files in os.walk(directory):
        print(f"Searching in subfolder: {root}")
        for file in files:
            if file.lower().endswith('.mp4'):
                full_path = os.path.join(root, file)
                mp4_files.append(full_path)
                print(f"Found MP4: {full_path}")
    return mp4_files

# Collect MP4 paths for each class with debugging
non_violent_mp4s = find_mp4_files(NON_VIOLENT_DIR)
violent_mp4s = find_mp4_files(VIOLENT_DIR)

print(f"\nSummary:")
print(f"Found {len(non_violent_mp4s)} non-violent MP4s: {non_violent_mp4s[:5]}")  # First 5 for brevity
print(f"Found {len(violent_mp4s)} violent MP4s: {violent_mp4s[:5]}")

# Assign labels
non_violent_data = [(path, 0) for path in non_violent_mp4s]
violent_data = [(path, 1) for path in violent_mp4s]
all_data = non_violent_data + violent_data

# Check if data is empty
if not all_data:
    raise ValueError("No MP4 files found! Verify: 1) Absolute path in DATA_ROOT (Section 2). 2) Folder names (e.g., 'non-violent' exact match). 3) Run !ls {NON_VIOLENT_DIR}/cam1 in a cell to check files. 4) Ensure files end with .mp4 (case-insensitive).")

# Unpack and split (stratified by label for balance)
paths, labels = zip(*all_data)

# Split into train + temp (85% train+val, 15% test)
train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
    paths, labels, test_size=0.15, stratify=labels, random_state=42
)

# Split train_val into train and val (approx 70/15/15 overall)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths, train_val_labels, test_size=0.1765, stratify=train_val_labels, random_state=42
)

print(f"Train samples: {len(train_paths)}, Val samples: {len(val_paths)}, Test samples: {len(test_paths)}")

Searching in subfolder: /home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/non-violent
Searching in subfolder: /home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/non-violent/cam2
Found MP4: /home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/non-violent/cam2/12.mp4
Found MP4: /home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/non-violent/cam2/51.mp4
Found MP4: /home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/non-violent/cam2/6.mp4
Found MP4: /home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/non-violent/cam2/40.mp4
Found MP4: /home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/non-violent/cam2/49.mp4
Found MP4: /home/heytanix/Documents/Jain_PCL/PCL_repository/raw_data/violence-detection-dataset/non-violent/cam2/10.mp4
Found MP4: /home/heytanix/Document

In [ ]:
print(f"Original distribution: {len(non_violent_mp4s)} non-violent, {len(violent_mp4s)} violent")

# Option 1: Balance by undersampling majority class (violent)
violent_balanced = resample(violent_data, n_samples=len(non_violent_data), random_state=42)
all_data_balanced = non_violent_data + violent_balanced

# Unpack balanced data and split
paths_balanced, labels_balanced = zip(*all_data_balanced)

# Re-split with balanced data
train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
    paths_balanced, labels_balanced, test_size=0.15, stratify=labels_balanced, random_state=42
)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths, train_val_labels, test_size=0.1765, stratify=train_val_labels, random_state=42
)

print(f"Balanced - Train: {len(train_paths)}, Val: {len(val_paths)}, Test: {len(test_paths)}")

Original distribution: 120 non-violent, 230 violent
Balanced - Train: 167, Val: 37, Test: 36


### Define VideoDataset Class

This cell defines a custom PyTorch `Dataset` class called `VideoDataset`. This class is responsible for loading video data and its corresponding labels.

- `__init__`: Initializes the dataset with video file paths, labels, desired clip length, and any transformations to apply.
- `__len__`: Returns the total number of video samples in the dataset.
- `__getitem__`: This is the core method that loads and preprocesses a single video sample given its index.
    - It uses `Decord` to efficiently load the video.
    - It samples a fixed number of frames (`self.clip_len`) from the video, either uniformly if the video is long enough or by repeating frames if it's shorter.
    - The frames are converted to a PyTorch tensor and normalized to a range of [0, 1].
    - Optional transformations (`self.transform`) are applied to the video tensor.
    - It returns the processed video tensor and its corresponding label.

In [ ]:
class VideoDataset(Dataset):
    def __init__(self, video_paths, labels, clip_len=16, transform=None):
        self.video_paths = video_paths
        self.labels = labels
        self.clip_len = clip_len
        self.transform = transform

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        vid_path = self.video_paths[idx]
        label = self.labels[idx]

        # Load video with Decord
        vr = VideoReader(vid_path, ctx=cpu(0))
        total_frames = len(vr)

        # Sample frames uniformly
        if total_frames>=self.clip_len:
            indices = np.linspace(0, total_frames - 1, self.clip_len, dtype=int)
        else:
            indices = np.tile(np.arange(total_frames), self.clip_len // total_frames + 1)[:self.clip_len]

        frames = vr.get_batch(indices).asnumpy()  # Shape: (clip_len, H, W, C)

        # Convert to tensor (C, T, H, W) and normalize
        frames = torch.from_numpy(frames).permute(3, 0, 1, 2).float() / 255.0
        if self.transform:
            frames = self.transform(frames)

        return frames, label

### Define Transformations and Create Data Loaders

This cell defines the data transformations to be applied to the video frames and creates the dataset and data loader instances.

- **Transformations:**
    - `transforms.Resize((112, 112))`: Resizes each frame to a standard size of 112x112 pixels, which is the expected input size for the R3D-18 model.
    - `transforms.Normalize(...)`: Normalizes the pixel values of the frames using the specified mean and standard deviation. These values are typical for datasets like ImageNet or Kinetics, which the R3D-18 model was likely pre-trained on.

- **Dataset Creation:**
    - `VideoDataset(...)`: Instances of the custom `VideoDataset` are created for the training, validation, and testing sets using the video paths and labels obtained from the data splitting step.

- **Data Loaders:**
    - `DataLoader(...)`: PyTorch `DataLoader` instances are created for each dataset. Data loaders provide an iterable over the dataset and handle batching, shuffling (for the training set), and parallel loading. The `batch_size` is set to 8, but can be adjusted based on available GPU memory.

Finally, the number of samples in each dataset split is printed to confirm the data loading and splitting process.

### Model Initialization, Loss Function, and Optimizer

This cell sets up the neural network model, defines the loss function, and configures the optimizer for training.

- **Device Configuration:** It checks if a CUDA-enabled GPU is available and sets the `device` to 'cuda' if it is, otherwise it uses the 'cpu'.
- **Model Loading and Modification:**
    - `r3d_18(pretrained=True)`: Loads the R3D-18 model with pre-trained weights (likely from Kinetics-400). This leverages knowledge learned from a large video dataset.
    - `model.fc = nn.Linear(model.fc.in_features, 2)`: The original R3D-18 model is designed for a different number of classes (400 for Kinetics). This line replaces the final fully connected layer (`fc`) with a new linear layer that outputs 2 values, corresponding to the two classes in this binary classification task (violent/non-violent).
    - `model.to(device)`: Moves the model to the selected device (GPU or CPU).
- **Loss Function:**
    - `nn.CrossEntropyLoss()`: This is the standard loss function for multi-class classification problems. It calculates the cross-entropy between the predicted probabilities and the true labels.
- **Optimizer:**
    - `optim.Adam(model.parameters(), lr=0.001)`: The Adam optimizer is chosen to update the model's weights during training. The learning rate (`lr`) is initially set to 0.001 and can be adjusted. The optimizer is configured to optimize all the parameters of the model.

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load pre-trained 3D ResNet and modify for 2 classes
model = r3d_18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)  # Binary classification
model.to(device)

# Weighted loss to handle any remaining class imbalance
class_counts = [len([l for l in train_labels if l == 0]), len([l for l in train_labels if l == 1])]
weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
weights = weights / weights.sum() * 2  # Normalize
criterion = nn.CrossEntropyLoss(weight=weights.to(device))

# Better optimizer with different learning rates
backbone_params = [p for n, p in model.named_parameters() if 'fc' not in n]
classifier_params = [p for n, p in model.named_parameters() if 'fc' in n]

optimizer = optim.Adam([
    {'params': backbone_params, 'lr': 0.0001},   # Lower LR for pre-trained features
    {'params': classifier_params, 'lr': 0.001}    # Higher LR for new classifier
])

# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

print(f"Class distribution in training: {class_counts}")
print(f"Class weights: {weights}")

Using device: cuda


/home/heytanix/Documents/Jain_PCL/PCL_repository/venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/heytanix/Documents/Jain_PCL/PCL_repository/venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=R3D_18_Weights.KINETICS400_V1`. You can also use `weights=R3D_18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Class distribution in training: [83, 84]
Class weights: tensor([1.0060, 0.9940])


### Custom Video Transformations (Resize and Normalize)

This cell defines custom transformation classes `ResizeVideo` and `NormalizeVideo` that are specifically designed to work with video tensors (which have a time dimension). While `torchvision.transforms` primarily works on single images, these classes apply the transformations consistently across all frames of a video clip.

- **`ResizeVideo`**:
    - Takes a target size as input.
    - The `__call__` method iterates through each frame of the input video tensor (`video: (C, T, H, W)`).
    - For each frame, it uses `torch.nn.functional.interpolate` with `mode='bilinear'` to resize the frame to the target size.
    - The resized frames are collected into a new video tensor.

- **`NormalizeVideo`**:
    - Takes mean and standard deviation values as input.
    - It reshapes the mean and standard deviation tensors to have a shape `(C, 1, 1)` so they can be correctly broadcasted across the time, height, and width dimensions when performing the normalization.
    - The `__call__` method applies the standard normalization formula `(video - mean) / std` to the entire video tensor.

Finally, an updated `transforms.Compose` object is created using these custom classes. This composition will be used when creating the `VideoDataset` instances to preprocess the video data.

In [ ]:
# Custom resize class for videos (applies to each frame)
class ResizeVideo:
    def __init__(self, size):
        self.size = size

    def __call__(self, video):
        # video: (C, T, H, W)
        c, t, h, w = video.shape
        video_resized = torch.zeros((c, t, self.size, self.size), dtype=video.dtype)
        for i in range(t):
            frame = video[:, i, :, :].unsqueeze(0)  # (1, C, H, W)
            frame_resized = F.interpolate(frame, size=(self.size, self.size), mode='bilinear', align_corners=False)
            video_resized[:, i, :, :] = frame_resized.squeeze(0)
        return video_resized

# Custom normalize class for videos (applies to each frame)
class NormalizeVideo:
    def __init__(self, mean, std):
        self.mean = torch.tensor(mean).view(-1, 1, 1)
        self.std = torch.tensor(std).view(-1, 1, 1)

    def __call__(self, video):
        # video: (C, T, H, W)
        return (video - self.mean[..., None, None]) / self.std[..., None, None]

# Updated transforms composition
transform = transforms.Compose([
    ResizeVideo(112),  # Resize each frame to 112x112
    NormalizeVideo(mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225])  # Normalize per channel across frames
])

### Corrected Custom Video Transformations

This cell provides a corrected version of the custom `NormalizeVideo` class.

- **`ResizeVideo`**: Remains the same as the previous cell, correctly resizing each frame individually.

- **`NormalizeVideo` (Corrected)**:
    - Takes mean and standard deviation values.
    - It reshapes the mean and standard deviation tensors to have a shape `(C, 1, 1, 1)`. This shape is crucial for correctly broadcasting the mean and standard deviation across the time (T), height (H), and width (W) dimensions of the video tensor `(C, T, H, W)`. This ensures that normalization is applied channel-wise across all frames and pixels.
    - The `__call__` method applies the normalization using the correctly shaped mean and standard deviation.

Finally, the `transforms.Compose` is updated to use the corrected `NormalizeVideo` class.

In [ ]:
# Custom resize for videos (resizes each frame individually)
class ResizeVideo:
    def __init__(self, size):
        self.size = size

    def __call__(self, video):
        # video: (C, T, H, W)
        c, t, h, w = video.shape
        video_resized = torch.zeros((c, t, self.size, self.size), dtype=video.dtype)
        for i in range(t):
            frame = video[:, i, :, :].unsqueeze(0)  # (1, C, H, W)
            frame_resized = F.interpolate(frame, size=(self.size, self.size), mode='bilinear', align_corners=False)
            video_resized[:, i, :, :] = frame_resized.squeeze(0)
        return video_resized

# Custom normalize for videos (broadcasts mean/std across T, H, W)
class NormalizeVideo:
    def __init__(self, mean, std):
        self.mean = torch.tensor(mean).view(-1, 1, 1, 1)  # Shape for broadcasting: (C, 1, 1, 1)
        self.std = torch.tensor(std).view(-1, 1, 1, 1)

    def __call__(self, video):
        # video: (C, T, H, W)
        return (video - self.mean) / self.std

# Updated transforms
transform = transforms.Compose([
    ResizeVideo(112),  # Resize to 112x112 per frame
    NormalizeVideo(mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225])  # Normalize across all frames
])

In [ ]:
class VideoAugmentation:
    def __init__(self, brightness=0.2, contrast=0.2, hue=0.1):
        self.brightness = brightness
        self.contrast = contrast
        self.hue = hue

    def __call__(self, video):
        # Random brightness
        if torch.rand(1)>0.5:
            factor = 1 + (torch.rand(1) - 0.5) * 2 * self.brightness
            video = video * factor

        # Random horizontal flip
        if torch.rand(1)>0.5:
            video = torch.flip(video, dims=[3])  # Flip width dimension

        return torch.clamp(video, 0, 1)

# Updated transforms with augmentation
transform_train = transforms.Compose([
    ResizeVideo(112),
    VideoAugmentation(),  # Only for training
    NormalizeVideo(mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225])
])

transform_val_test = transforms.Compose([
    ResizeVideo(112),
    NormalizeVideo(mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225])
])


### Create Dataset and Data Loaders

This cell creates instances of the `VideoDataset` and `DataLoader` classes for the training, validation, and testing sets.

- `VideoDataset(...)`: Instances of the custom `VideoDataset` are created for the training, validation, and testing sets using the video paths and labels obtained from the data splitting step. Different transformations (`transform_train` and `transform_val_test`) are applied to the training set compared to the validation and test sets to incorporate data augmentation only during training. The `clip_len` parameter specifies the number of frames to sample from each video.
- `DataLoader(...)`: PyTorch `DataLoader` instances are created for each dataset. Data loaders provide an iterable over the dataset and handle batching, shuffling (for the training set), and parallel loading. The `batch_size` is set to 8, but can be adjusted based on available GPU memory. `shuffle=True` is used for the training loader to randomize the order of samples in each epoch, while `shuffle=False` is used for validation and test loaders to maintain a fixed order.

Finally, the number of samples in each dataset split is printed to confirm the data loading and splitting process.

In [ ]:
# Create datasets with different transforms for train vs val/test
train_dataset = VideoDataset(train_paths, train_labels, clip_len=16, transform=transform_train)
val_dataset = VideoDataset(val_paths, val_labels, clip_len=16, transform=transform_val_test)
test_dataset = VideoDataset(test_paths, test_labels, clip_len=16, transform=transform_val_test)

In [ ]:
# Data loaders
batch_size = 8  # Adjust based on GPU memory
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}, Test samples: {len(test_dataset)}")

Train samples: 167, Val samples: 37, Test samples: 36


### Define Training Function

This cell defines the `train_model` function, which encapsulates the entire training and validation loop.

- **Parameters:**
  - `model`: The PyTorch model to be trained.
  - `train_loader`: DataLoader for the training data.
  - `val_loader`: DataLoader for the validation data.
  - `criterion`: The loss function.
  - `optimizer`: The optimizer for updating model weights.
  - `scheduler`: The learning rate scheduler.
  - `num_epochs`: The total number of training epochs.
- **Training Loop:**
  - The function iterates through the specified number of epochs.
  - In each epoch, it performs a training phase and a validation phase.
  - **Training Phase (`model.train()`):**
    - Sets the model to training mode.
    - Iterates through batches from the `train_loader`.
    - Moves data and labels to the appropriate device (`cuda` or `cpu`).
    - Zeros the gradients of the optimizer.
    - Performs a forward pass (`model(videos)`).
    - Calculates the loss (`criterion(outputs, labels)`).
    - Performs a backward pass (`loss.backward()`) to compute gradients.
    - Updates model weights (`optimizer.step()`).
    - Tracks the running training loss and calculates training accuracy.
    - Prints progress updates periodically.
  - **Validation Phase (`model.eval()`, `with torch.no_grad():`)**:
    - Sets the model to evaluation mode (disables dropout, batch normalization statistics fixed).
    - Disables gradient calculation for efficiency.
    - Iterates through batches from the `val_loader`.
    - Moves data and labels to the device.
    - Performs a forward pass and calculates the validation loss.
    - Calculates validation accuracy.
- **Learning Rate Scheduling:**
  - `scheduler.step()` is called after each epoch to update the learning rate based on the configured schedule.
- **Model Saving:**
  - The function keeps track of the model state with the best validation accuracy and saves it.
- **Output and Plotting:**
  - Prints epoch-wise training and validation metrics (loss and accuracy).
  - Plots the training loss and validation accuracy over epochs using `matplotlib`.
  - Loads the state dictionary of the best performing model based on validation accuracy.
- **Return Values:**
  - Returns lists of training losses and validation accuracies for plotting.

The cell then calls the `train_model` function to start the training process and displays the resulting plots.

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=3):
    """Complete training function with validation and monitoring"""

    train_losses = []
    val_accuracies = []
    best_val_acc = 0.0
    best_model_state = None

    print(f"Starting training for {num_epochs} epochs...")
    print(f"Training batches: {len(train_loader)}, Validation batches: {len(val_loader)}")

    for epoch in range(num_epochs):
        epoch_start = datetime.now()

        # Training phase
        model.train()
        running_loss = 0.0
        train_correct = 0
        train_total = 0

        for batch_idx, (videos, labels) in enumerate(train_loader):
            videos, labels = videos.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(videos)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

            # Print progress every 10 batches
            if (batch_idx + 1) % 10 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx+1}/{len(train_loader)}], '
                      f'Loss: {loss.item():.4f}')

        # Validation phase
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0.0

        with torch.no_grad():
            for videos, labels in val_loader:
                videos, labels = videos.to(device), labels.to(device)
                outputs = model(videos)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        # Calculate metrics
        train_acc = 100 * train_correct / train_total
        val_acc = 100 * val_correct / val_total
        avg_train_loss = running_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        train_losses.append(avg_train_loss)
        val_accuracies.append(val_acc)

        # Update learning rate
        scheduler.step()

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()

        epoch_time = datetime.now() - epoch_start

        print(f'Epoch [{epoch+1}/{num_epochs}] ({epoch_time.total_seconds():.1f}s):')
        print(f'  Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%')
        print(f'  Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%')
        print(f'  LR: {optimizer.param_groups[0]["lr"]:.6f}')
        print('-' * 60)

    print(f'Training completed! Best validation accuracy: {best_val_acc:.2f}%')

    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print("Loaded best model weights")

    return train_losses, val_accuracies

# Run the training!
print("="*80)
print("="*80)

train_losses, val_accuracies = train_model(
    model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=25
)

# Plot training progress
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.subplot(1, 2, 2)
plt.plot(val_accuracies)
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')

plt.tight_layout()
plt.show()

print("Training completed!")


STARTING TRAINING - THIS IS WHAT WAS MISSING!
Starting training for 25 epochs...
Training batches: 21, Validation batches: 5


### Save Trained Model State

This cell saves the state dictionary of the trained model to a file. This allows you to load the trained model later for inference or further training without needing to retrain it from scratch.

- `torch.save(model.state_dict(), 'violence_classifier.pth')`: This line uses the `torch.save` function to serialize the model's `state_dict()` (a Python dictionary containing the model's learned parameters like weights and biases) and saves it to a file named `violence_classifier.pth` in the current working directory.
- A confirmation message is printed to indicate that the model state has been successfully saved.

In [ ]:
torch.save(model.state_dict(), 'violence_classifier.pth')
print("Model saved successfully!")

Model saved successfully!
